[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ohtopico/site/blob/main/notebooks/wikipedia/01-do-zero-ao-grafico.ipynb)

# Do zero ao gráfico: visitas da Wikipédia com Python

Toda visita a um verbete da Wikipédia é contada, e a Wikimedia publica esses números como dados abertos. Neste notebook você escolhe verbetes, baixa as visitas mês a mês e monta gráficos com título, eixos e fonte, como em um relatório de pesquisa.

**Público:** professores, 9º ano, ensino médio e cursos técnicos · **Tempo:** 1 aula de 50 min · **BNCC:** EF07MA36, EF09MA22, EF08CO11

Notebook de [Júlia Roberta Quoos Alves](https://ohtopico.com.br/curriculo/) · [ohtopico.com.br](https://ohtopico.com.br/interesse-por-ia/notebooks/) · Licença CC BY-NC-SA 4.0

**Como usar:** clique em cada célula de código e aperte o botão de play (ou Shift + Enter), de cima para baixo. Os campos ao lado das células (listas e caixas de texto) podem ser trocados à vontade: depois de mudar, rode a célula de novo.

## Objetivos
- Coletar dados abertos por meio de uma API.
- Organizar uma série temporal em tabela (um mês por linha, um verbete por coluna).
- Escolher e construir o gráfico adequado: linhas para dados no tempo, colunas para comparar totais.

## Passo 0. Configuração
Rode a célula abaixo uma vez. Ela carrega as ferramentas e cria a função `visitas()`, que busca os números na Wikimedia.

In [ ]:
# @title Configuração (rode primeiro) { display-mode: "form" }
# Funções para baixar os dados públicos da Wikimedia. Não precisa mudar nada aqui.
import json, re, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

CORES = ['#157f3b', '#b3123a', '#c99a00', '#111111', '#5fb27c', '#7a1f33']
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.prop_cycle': plt.cycler(color=CORES), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 11})
pd.options.display.float_format = '{:,.0f}'.format
UA = {'User-Agent': 'notebooks-ohtopico/1.0 (https://ohtopico.com.br; contato@ohtopico.com.br)'}
FONTE = 'Fonte: Wikimedia, visitas de pessoas à Wikipédia em português'


def _get(url, params=None):
    r = requests.get(url, params=params, headers=UA, timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()


def titulo_oficial(titulo):
    """Corrige maiúsculas e segue redirecionamentos: 'ia' vira 'Inteligência artificial'."""
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', titles=titulo, redirects=1, format='json', formatversion=2))
    pagina = j['query']['pages'][0]
    if pagina.get('missing'):
        raise ValueError(f'Não achei o verbete "{titulo}" na Wikipédia em português. Confira a grafia.')
    return pagina['title']


def visitas(titulos, inicio='2016-01-01', fim=None, por='mes'):
    """Tabela com as visitas de cada verbete. por='mes' ou por='dia'."""
    if isinstance(titulos, str):
        titulos = [titulos]
    fim = pd.Timestamp(fim) if fim else pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    if por == 'mes':
        fim = fim.replace(day=1) - pd.Timedelta(days=1)  # só meses completos
    colunas = {}
    for t in titulos:
        t = titulo_oficial(t)
        art = urllib.parse.quote(t.replace(' ', '_'), safe='')
        g = 'monthly' if por == 'mes' else 'daily'
        url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/pt.wikipedia/all-access/user/{art}/{g}/'
               f'{pd.Timestamp(inicio):%Y%m%d}00/{fim:%Y%m%d}00')
        j = _get(url) or {'items': []}
        colunas[t] = pd.Series({pd.Timestamp(i['timestamp'][:8]): i['views'] for i in j['items']}, dtype='float64')
        time.sleep(.1)
    df = pd.DataFrame(colunas).sort_index()
    idx = pd.date_range(pd.Timestamp(inicio), fim, freq='MS' if por == 'mes' else 'D')
    df = df.reindex(idx).fillna(0).astype(int)
    df.index.name = 'mês' if por == 'mes' else 'dia'
    return df


def texto(titulo):
    """Texto puro de um verbete (sem tabelas nem imagens)."""
    t = titulo_oficial(titulo)
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', prop='extracts', explaintext=1, titles=t,
                                                         format='json', formatversion=2))
    return j['query']['pages'][0].get('extract', '')


def fmt(v, casas=0):
    """Número no formato brasileiro: 1.234,5"""
    return f'{v:,.{casas}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


def baixar_csv(df, nome):
    df.to_csv(nome)
    try:
        from google.colab import files
        files.download(nome)
    except ImportError:
        print('Arquivo salvo:', nome)


print('Pronto. Funções disponíveis: visitas(), texto(), titulo_oficial(), fmt(), baixar_csv().')

## Passo 1. Escolha os verbetes
Escreva o nome de até três verbetes da Wikipédia em português, do jeito que aparecem no título da página. Pode trocar por qualquer tema da turma: um time, um planeta, um jogo, uma festa.

In [ ]:
verbete_1 = 'Inteligência artificial'  # @param {type:"string"}
verbete_2 = 'ChatGPT'  # @param {type:"string"}
verbete_3 = 'Robótica'  # @param {type:"string"}
desde = '2018-01-01'  # @param {type:"date"}

escolhidos = [v for v in (verbete_1, verbete_2, verbete_3) if v.strip()]
df = visitas(escolhidos, inicio=desde)
print(f'{len(df)} meses, de {df.index[0]:%m/%Y} a {df.index[-1]:%m/%Y}')
df.tail()

## Passo 2. Conheça a tabela
Cada linha é um mês e cada coluna é um verbete. O resumo abaixo mostra a média, o menor e o maior valor de cada coluna.

**Para discutir:** a média é um bom resumo para esses dados? O que acontece com a média quando há um mês com um pico enorme?

In [ ]:
df.describe().loc[['mean', 'min', '50%', 'max']].rename(index={'mean': 'média', 'min': 'mínimo', '50%': 'mediana', 'max': 'máximo'})

## Passo 3. Gráfico de linhas
Dados ao longo do tempo pedem gráfico de linhas. Repare nos elementos que todo gráfico precisa ter: título, nome dos eixos, legenda e fonte.

In [ ]:
ax = df.plot(linewidth=2.2)
ax.set_title('Visitas por mês na Wikipédia em português')
ax.set_xlabel('Mês')
ax.set_ylabel('Visitas')
ax.yaxis.set_major_formatter(lambda v, p: f'{v:,.0f}'.replace(',', '.'))
ax.legend(title='Verbete')
plt.figtext(0.01, -0.02, FONTE, fontsize=9, color='#4a4a4a')
plt.show()

## Passo 4. Totais por ano
Para comparar anos inteiros, somamos os meses de cada ano e usamos colunas. O ano atual aparece incompleto: lembre disso ao comparar.

In [ ]:
por_ano = df.groupby(df.index.year).sum()
ax = por_ano.plot(kind='bar', width=.8, edgecolor='#111')
ax.set_title('Visitas por ano')
ax.set_xlabel('Ano')
ax.set_ylabel('Visitas')
ax.yaxis.set_major_formatter(lambda v, p: f'{v:,.0f}'.replace(',', '.'))
plt.xticks(rotation=0)
plt.figtext(0.01, -0.02, FONTE + '. O último ano está incompleto.', fontsize=9, color='#4a4a4a')
plt.show()
por_ano

## Passo 5. Explore
Escolha um verbete e o tipo de escala. A escala logarítmica ajuda quando um verbete tem números muito maiores que os outros: cada linha da grade vale dez vezes a anterior.

In [ ]:
@interact(verbete=list(df.columns), escala=['comum', 'logarítmica'], media_movel=widgets.IntSlider(value=1, min=1, max=12, description='média móvel'))
def explorar(verbete, escala, media_movel):
    s = df[verbete]
    ax = s.plot(alpha=.35 if media_movel > 1 else 1, label='visitas')
    if media_movel > 1:
        s.rolling(media_movel).mean().plot(ax=ax, linewidth=2.5, label=f'média dos últimos {media_movel} meses')
    if escala == 'logarítmica':
        ax.set_yscale('log')
    ax.set_title(f'{verbete}: visitas por mês')
    ax.legend()
    plt.show()

## Sua vez
Em que mês cada verbete teve mais visitas, e quantas foram? Escreva o código na célula abaixo. Dica: `df.idxmax()` e `df.max()`.

In [ ]:
# escreva aqui

<details><summary><b>Ver uma solução</b></summary>

```python
recorde = pd.DataFrame({'mês do recorde': df.idxmax().dt.strftime('%m/%Y'), 'visitas': df.max()})
recorde
```
</details>

## Guardar os dados
A célula abaixo salva a tabela em CSV, que abre em qualquer planilha.

In [ ]:
baixar_csv(df, 'visitas-wikipedia.csv')

## Para fechar
- O que os gráficos mostram e o que eles **não** mostram? (Visitas medem curiosidade, não uso nem opinião.)
- A Wikipédia em português é lida no Brasil, em Portugal e em outros países. Isso muda as conclusões?
- Escolha o maior pico e procure em duas fontes confiáveis o que aconteceu naquele mês.

Continue com o notebook [Detecção de picos](https://colab.research.google.com/github/ohtopico/site/blob/main/notebooks/wikipedia/03-deteccao-de-picos.ipynb) ou veja o [painel de interesse por IA](https://ohtopico.com.br/interesse-por-ia/).